## create a connection to the database using the library sqlite3

In [74]:
import pandas as pd
import sqlite3
connection = sqlite3.connect('../../data/checking-logs.sqlite')
pd.io.sql.read_sql('PRAGMA table_info(test);', connection)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [75]:
pd.io.sql.read_sql('SELECT * FROM test LIMIT 10', connection)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## find among all the users the minimum value

In [76]:
query = """SELECT
t.uid,
MIN((STRFTIME('%s', t.first_commit_ts) - dl.deadlines) / 3600) AS min_val
FROM test AS t
INNER JOIN deadlines AS dl
    ON dl.labs = t.labname
WHERE t.labname != 'project1';
        """

df_min = pd.io.sql.read_sql(query, connection)

In [77]:
df_min

,uid,min_val
0,user_30,-202


In [78]:
query = """SELECT
t.uid,
MAX((STRFTIME('%s', t.first_commit_ts) - dl.deadlines) / 3600) AS min_val
FROM test AS t
INNER JOIN deadlines AS dl
    ON dl.labs = t.labname
WHERE t.labname != 'project1';
        """

df_max = pd.io.sql.read_sql(query, connection)

In [79]:
df_max

,uid,min_val
0,user_25,-2


In [80]:
query = """SELECT
    AVG((STRFTIME('%s', t.first_commit_ts) - dl.deadlines) / 3600) AS avg_val
FROM test AS t
INNER JOIN deadlines AS dl
    ON t.labname = dl.labs
WHERE t.labname != 'project1';
        """
df_avg = pd.io.sql.read_sql(query, connection)

In [81]:
df_avg

,avg_val
0,-89.125


## we want to test the hypothesis

In [82]:
query = """WITH base AS (
    SELECT
        t.uid AS user_id,
        CAST((strftime('%s', t.first_commit_ts) - dl.deadlines) / 3600 AS INTEGER) AS avg_diff_hours
    FROM
        test AS t
        INNER JOIN deadlines AS dl
        ON  t.labname = dl.labs
        AND t.labname != 'project1')
SELECT
    b.user_id AS uid,
    b.avg_diff_hours AS avg_diff,
    COUNT(DISTINCT pv.datetime) AS pageviews
FROM
    base AS b
    LEFT JOIN pageviews AS pv
    ON pv.uid = b.user_id
GROUP BY
    b.user_id;
        """
views_diff = pd.io.sql.read_sql(query, connection)

In [83]:
views_diff

,uid,avg_diff,pageviews
0,user_1,-6,28
1,user_10,-39,89
2,user_14,-200,143
3,user_17,-81,47
4,user_18,-4,3
5,user_19,-148,16
6,user_21,-126,10
7,user_25,-148,179
8,user_28,-98,149
9,user_3,-75,317


In [84]:
views_diff.corr(numeric_only=True)

,avg_diff,pageviews
avg_diff,1.000000,-0.062967
pageviews,-0.062967,1.000000


In [85]:
connection.close()